In [1]:
import duckdb

# Establish a connection to an in-memory DuckDB instance
con = duckdb.connect(database=':memory:')

# Write the SQL query to inspect the data directly from the CSV
query = """
    SELECT 
        SK_ID_CURR, 
        TARGET, 
        NAME_CONTRACT_TYPE, 
        CODE_GENDER, 
        AMT_INCOME_TOTAL
    FROM read_csv_auto('data/raw/application_train.csv')
    LIMIT 5;
"""

# Execute the query and fetch the result as a pandas DataFrame
df_sample = con.execute(query).df()

# Display the result
display(df_sample)

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,AMT_INCOME_TOTAL
0,100002,1,Cash loans,M,202500.0
1,100003,0,Cash loans,F,270000.0
2,100004,0,Revolving loans,M,67500.0
3,100006,0,Cash loans,F,135000.0
4,100007,0,Cash loans,M,121500.0


In [4]:
# Query to describe the structure of the CSV
schema_query = """
    DESCRIBE SELECT * FROM read_csv_auto('data/raw/application_train.csv');
"""

# Fetch and display schema
df_schema = con.execute(schema_query).df()
display(df_schema)

,column_name,column_type,null,key,default,extra
0,SK_ID_CURR,BIGINT,YES,None,None,None
1,TARGET,BIGINT,YES,None,None,None
2,NAME_CONTRACT_TYPE,VARCHAR,YES,None,None,None
3,CODE_GENDER,VARCHAR,YES,None,None,None
4,FLAG_OWN_CAR,VARCHAR,YES,None,None,None
...,...,...,...,...,...,...
117,AMT_REQ_CREDIT_BUREAU_DAY,DOUBLE,YES,None,None,None
118,AMT_REQ_CREDIT_BUREAU_WEEK,DOUBLE,YES,None,None,None
119,AMT_REQ_CREDIT_BUREAU_MON,DOUBLE,YES,None,None,None
120,AMT_REQ_CREDIT_BUREAU_QRT,DOUBLE,YES,None,None,None


In [6]:
# Filter the df by column name
display(df_schema[df_schema['column_name'] == 'AMT_INCOME_TOTAL'])

,column_name,column_type,null,key,default,extra
7,AMT_INCOME_TOTAL,DOUBLE,YES,None,None,None


In [12]:
# Count total rows and missing values for specific columns
null_query = """
    SELECT 
        COUNT(*) AS total_applicants,
        COUNT(*) - COUNT(AMT_INCOME_TOTAL) AS missing_income,
        COUNT(*) - COUNT(EXT_SOURCE_1) AS missing_ext_source_1,
        COUNT(*) - COUNT(EXT_SOURCE_2) AS missing_ext_source_2,
        COUNT(*) - COUNT(EXT_SOURCE_3) AS missing_ext_source_3
    FROM read_csv_auto('data/raw/application_train.csv');
"""

# Execute and display the result
df_nulls = con.execute(null_query).df()
display(df_nulls)

,total_applicants,missing_income,missing_ext_source_1,missing_ext_source_2,missing_ext_source_3
0,307511,0,173378,660,60965


In [13]:
# Query to calculate default rates based on missing data
missingness_query = """
    SELECT
        CASE 
            WHEN EXT_SOURCE_1 IS NULL THEN 'Missing Score'
            ELSE 'Has Score'
        END AS ext_source_1_status,
        COUNT(*) AS total_applicants,
        AVG(TARGET) AS default_rate
    FROM read_csv_auto('data/raw/application_train.csv')
    GROUP BY 
        ext_source_1_status;
"""

# Execute and display the result
df_missingness = con.execute(missingness_query).df()
display(df_missingness)

,ext_source_1_status,total_applicants,default_rate
0,Has Score,134133,0.074955
1,Missing Score,173378,0.085195
